In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

In [2]:
df_channel = pd.read_excel("D:/数据分析求职/channel.xlsx")

In [3]:
#定义权重
weights = {
    'repeat_rate_pct': 0.3,      # 复购最重要
    'avg_ltv': 0.3,          # LTV同样重要
    'avg_lifetime_days': 0.2,
    'overall_aov': 0.1,
    'first_order_aov': 0.1}

In [4]:
#采用Minmaxscaler进行标准化
scaler = MinMaxScaler()
metrics = ['repeat_rate_pct', 'first_order_aov', 'overall_aov', 'avg_lifetime_days', 'avg_ltv']
df_channel_scaled = df_channel.copy()
df_channel_scaled[metrics] = scaler.fit_transform(df_channel[metrics])
df_channel_scaled['quality_score'] = sum(df_channel_scaled[m] * w for m, w in weights.items())
df_channel_scaled.sort_values('quality_score', ascending=False)

,acquisition_channel,new_customers,repeat_rate_pct,first_order_aov,overall_aov,avg_lifetime_days,avg_ltv,quality_score
0,Paid Ad,1213,1.000000,0.340550,0.000000,1.000000,1.000000,0.834055
2,Organic Search,2084,0.658402,1.000000,0.476190,0.348984,0.883122,0.679873
1,Direct,786,0.421488,0.188177,1.000000,0.213560,0.888723,0.554593
3,Email Campaign,1436,0.347107,0.235637,0.403941,0.114751,0.434653,0.321436
4,Referral,458,0.000000,0.000000,0.068966,0.478802,0.390963,0.219946
5,Social Media,1683,0.374656,0.420483,0.098522,0.000000,0.000000,0.164297


In [7]:
#采用standardscaler进行标准化
scaler = StandardScaler()
df_channel_scaled1 = df_channel.copy()
df_channel_scaled1[metrics] = scaler.fit_transform(df_channel[metrics])
df_channel_scaled1['quality_score'] = sum(df_channel_scaled1[m] * w for m, w in weights.items())
df_channel_scaled1.sort_values('quality_score', ascending=False)

,acquisition_channel,new_customers,repeat_rate_pct,first_order_aov,overall_aov,avg_lifetime_days,avg_ltv,quality_score
0,Paid Ad,1213,1.738973,-0.075352,-0.995165,1.969254,1.129145,1.147235
2,Organic Search,2084,0.624592,2.030965,0.393437,-0.031862,0.799562,0.663314
1,Direct,786,-0.148285,-0.562039,1.920899,-0.448133,0.815357,0.246381
3,Email Campaign,1436,-0.390932,-0.410448,0.182753,-0.751854,-0.465066,-0.429940
4,Referral,458,-1.523287,-1.163084,-0.794057,0.367176,-0.588265,-0.755744
5,Social Media,1683,-0.301063,0.179959,-0.707868,-1.104581,-1.690734,-0.871246


In [8]:
#采用百分位排名法
def channel_quality_scoring(df_channel, metrics, weights):
    df = df_channel.copy()
    # 计算每个指标的百分位排名
    for metric in metrics:
        df[f'{metric}_rank'] = df[metric].rank(pct=True) * 100
    # 计算综合得分
    df['quality_score'] = 0
    for metric, weight in weights.items():
        df['quality_score'] += df[f'{metric}_rank'] * weight
    # 添加等级评级
    df['quality_grade'] = pd.cut(df['quality_score'], bins=[0, 25, 50, 75, 100],labels=['D', 'C', 'B', 'A'])
    return df.sort_values('quality_score', ascending=False)
df_channel_scored = channel_quality_scoring(df_channel, metrics, weights)
print("渠道质量排名：")
print(df_channel_scored[['acquisition_channel'] + [f'{m}_rank' for m in metrics] + [ 'quality_score', 'quality_grade']])

渠道质量排名：
  acquisition_channel  repeat_rate_pct_rank  first_order_aov_rank  \
0             Paid Ad            100.000000             66.666667   
2      Organic Search             83.333333            100.000000   
1              Direct             66.666667             33.333333   
3      Email Campaign             33.333333             50.000000   
4            Referral             16.666667             16.666667   
5        Social Media             50.000000             83.333333   

   overall_aov_rank  avg_lifetime_days_rank  avg_ltv_rank  quality_score  \
0         16.666667              100.000000    100.000000      88.333333   
2         83.333333               66.666667     66.666667      76.666667   
1        100.000000               50.000000     83.333333      68.333333   
3         66.666667               33.333333     50.000000      43.333333   
4         33.333333               83.333333     33.333333      36.666667   
5         50.000000               16.666667     16.6